# SNOMED CT Markdown Tagger

This notebook ingests one or more **Markdown (`.md`) files** plus an **official SNOMED CT RF2 release ZIP**, then emits **one CSV per input file** that lists every word in the document that matches a SNOMED CT concept, classified as one of:

| tag | definition |
| --- | --- |
| `disease` | descendants of `64572001 \| Disorder` |
| `clinical finding` | descendants of `404684003 \| Clinical finding` (excluding Disorder) |
| `procedure` | descendants of `71388002 \| Procedure` |
| `drug` | descendants of `373873005 \| Pharmaceutical / biologic product` **or** `105590001 \| Substance` |
| `clinical term` | any other SNOMED CT concept (body structure, organism, event, etc.) |

Each output CSV has columns `number, word, tag, page` plus `entity_id, concept_id` for traceability.

* **Pages** are defined by top-level Markdown headings (`#` or `##`). Each such heading starts a new page; everything before the first heading is page 1.
* **Multi-word concepts** (e.g. *myocardial infarction*) produce one row per word, all sharing the same `tag` and `entity_id`.
* `number` is a 1-based word index within the file and resets per file.

## How to run

Click *Runtime &rarr; Run all*, then when prompted upload (1) a SNOMED CT RF2 release ZIP and (2) one or more `.md` files. A `tagged_outputs.zip` will be downloaded at the end.

## License notice

SNOMED CT is a **licensed** terminology. You must hold a valid **SNOMED CT Affiliate License** (free in SNOMED International Member countries, fee-bearing in non-Member countries). Obtain release files from the [Member Licensing &amp; Distribution Service (MLDS)](https://mlds.ihtsdotools.org/) or your country's National Release Center. This notebook never downloads SNOMED CT on your behalf &mdash; you supply the release ZIP yourself.

In [ ]:
!pip install -q flashtext spacy pandas tqdm

## 1. Configuration

Top-level SNOMED CT concept IDs, tag priority order, and Colab paths.

In [ ]:
import os, re, io, sys, glob, zipfile, time, csv
from pathlib import Path
from collections import defaultdict, deque, Counter
from typing import Iterator, Iterable

# Top-level SNOMED CT concept IDs we care about, in priority order.
# 'disease' wins over 'clinical finding' because Disorder is a sub-hierarchy of Clinical finding.
# 'drug' includes both Pharmaceutical/biologic product AND Substance.
TAG_PRIORITY = [
    ("disease",          ["64572001"]),                # 64572001  | Disorder
    ("clinical finding", ["404684003"]),               # 404684003 | Clinical finding
    ("procedure",        ["71388002"]),                # 71388002  | Procedure
    ("drug",             ["373873005", "105590001"]),  # 373873005 | Pharmaceutical/biologic product
                                                       # 105590001 | Substance
]
DEFAULT_TAG = "clinical term"

# SNOMED CT TypeIds we care about
TYPE_FSN     = "900000000000003001"  # Fully Specified Name
TYPE_SYNONYM = "900000000000013009"  # Synonym
TYPE_IS_A    = "116680003"           # Is-a relationship

# Workdirs (Colab default is /content/). Override by setting these before re-running.
SNOMED_DIR = Path("/content/snomed")
MD_DIR     = Path("/content/markdown_inputs")
OUT_DIR    = Path("/content/output")
for _d in (SNOMED_DIR, MD_DIR, OUT_DIR):
    _d.mkdir(parents=True, exist_ok=True)

# Terms shorter than this are skipped during matching to reduce noise from one/two-letter codes.
MIN_TERM_LEN = 3

print("Configuration loaded.")
print(f"  Tags (priority order): {[t for t, _ in TAG_PRIORITY] + [DEFAULT_TAG]}")
print(f"  SNOMED unpack dir:     {SNOMED_DIR}")
print(f"  Markdown input dir:    {MD_DIR}")
print(f"  Output dir:            {OUT_DIR}")

## 2. SNOMED RF2 loader

Locates and parses the three Snapshot files we need from the unzipped release directory:

* `sct2_Concept_Snapshot_*.txt` &mdash; gives us the set of active concept IDs.
* `sct2_Description_Snapshot-en*.txt` &mdash; English FSNs and synonyms (the strings we match against).
* `sct2_Relationship_Snapshot_*.txt` &mdash; IS-A edges used to build the hierarchy.

All three are tab-separated files with a header row. We use a streaming `csv.DictReader` so even the ~1.5 GB Description file fits comfortably in Colab memory.

In [ ]:
def _iter_rf2(path: Path) -> Iterator[dict]:
    """Yield dict rows from a SNOMED RF2 tab-separated file (UTF-8, no quoting)."""
    with open(path, encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f, delimiter="\t", quoting=csv.QUOTE_NONE)
        for row in reader:
            yield row


def find_rf2_files(snomed_dir: Path) -> dict:
    """Locate the Snapshot Concept, Description, Relationship files anywhere under snomed_dir.

    If multiple files match a pattern (e.g. an Edition release that ships both International
    and country files), we pick the largest one, which is reliably the main Snapshot file.
    """
    patterns = {
        "concepts":      "sct2_Concept_Snapshot_*.txt",
        "descriptions":  "sct2_Description_Snapshot-en*.txt",
        "relationships": "sct2_Relationship_Snapshot_*.txt",
    }
    found = {}
    for name, pat in patterns.items():
        hits = sorted(snomed_dir.rglob(pat))
        if not hits:
            raise FileNotFoundError(
                f"Could not find {pat} anywhere under {snomed_dir}.\n"
                "Make sure you uploaded a SNAPSHOT (not Delta-only) SNOMED CT RF2 release ZIP\n"
                "containing English descriptions."
            )
        found[name] = max(hits, key=lambda p: p.stat().st_size)
    print("Located RF2 files:")
    for k, v in found.items():
        rel = v.relative_to(snomed_dir)
        print(f"  {k:14s} -> {rel}  ({v.stat().st_size / 1e6:,.1f} MB)")
    return found


def load_active_concept_ids(concept_file: Path) -> set:
    """Return the set of conceptIds with active=1."""
    out = set()
    for r in _iter_rf2(concept_file):
        if r.get("active") == "1":
            out.add(r["id"])
    print(f"Active concepts: {len(out):,}")
    return out


def iter_active_descriptions(
    desc_file: Path, active_concepts: set
) -> Iterator[tuple]:
    """Yield (conceptId, term, typeId) for every active English FSN or synonym whose
    concept is active. Streaming-only to avoid materialising ~1.4 M tuples."""
    for r in _iter_rf2(desc_file):
        if r.get("active") != "1":
            continue
        type_id = r.get("typeId")
        if type_id not in (TYPE_FSN, TYPE_SYNONYM):
            continue
        cid = r.get("conceptId")
        if cid not in active_concepts:
            continue
        term = r.get("term") or ""
        if not term:
            continue
        yield cid, term, type_id


def load_isa_edges(rel_file: Path, active_concepts: set) -> dict:
    """Return parent_id -> set(child_ids) for active IS-A edges between active concepts."""
    parent_to_children: dict = defaultdict(set)
    edge_count = 0
    for r in _iter_rf2(rel_file):
        if r.get("active") != "1":
            continue
        if r.get("typeId") != TYPE_IS_A:
            continue
        src = r.get("sourceId")        # child
        dst = r.get("destinationId")   # parent
        if src not in active_concepts or dst not in active_concepts:
            continue
        parent_to_children[dst].add(src)
        edge_count += 1
    print(
        f"Active IS-A edges: {edge_count:,}; "
        f"distinct parents: {len(parent_to_children):,}"
    )
    return parent_to_children

## 3. Hierarchy and tag classifier

Builds, via BFS over the IS-A graph, the set of descendants for each top-level concept we care about. Iterating `TAG_PRIORITY` in order and writing only if not already written gives us the documented priority: `disease > clinical finding > procedure > drug`. Anything not in the resulting map falls back to `clinical term` at match time.

In [ ]:
def compute_descendants(parent_to_children: dict, root_ids: Iterable[str]) -> set:
    """All descendants of any of root_ids, inclusive of the roots themselves."""
    seen: set = set()
    stack = list(root_ids)
    while stack:
        cur = stack.pop()
        if cur in seen:
            continue
        seen.add(cur)
        stack.extend(parent_to_children.get(cur, ()))
    return seen


def build_concept_to_tag(parent_to_children: dict) -> dict:
    """Return {concept_id: tag} for concepts under one of the 4 named tags, respecting
    TAG_PRIORITY (earlier-listed tags win on collision). Concepts not in the map fall
    through to DEFAULT_TAG at match time."""
    concept_to_tag: dict = {}
    tag_sizes: dict = {}
    for tag, roots in TAG_PRIORITY:
        descendants = compute_descendants(parent_to_children, roots)
        new_for_tag = 0
        for cid in descendants:
            if cid not in concept_to_tag:
                concept_to_tag[cid] = tag
                new_for_tag += 1
        tag_sizes[tag] = new_for_tag
    print("Concepts assigned per tag (priority-respecting):")
    for t, n in tag_sizes.items():
        print(f"  {t:18s} {n:>10,}")
    print(f"All other matched concepts default to: '{DEFAULT_TAG}'")
    return concept_to_tag

## 4. Matcher builder

`FlashText` is an Aho-Corasick-style keyword matcher that handles hundreds of thousands of phrases efficiently and returns longest, non-overlapping matches by default. We strip the trailing semantic tag from each Fully Specified Name (e.g. `"Heart failure (disorder)"` &rarr; `"Heart failure"`) so the matcher only sees terms that actually appear in clinical prose.

In [ ]:
from flashtext import KeywordProcessor

# 'Heart failure (disorder)' -> 'Heart failure'
# Only strips a single trailing parenthetical with no nested parens, which is the
# RF2 convention for semantic tags on Fully Specified Names.
_SEMANTIC_TAG_RE = re.compile(r"\s+\([^()]+\)\s*$")


def strip_semantic_tag(term: str) -> str:
    return _SEMANTIC_TAG_RE.sub("", term).strip()


def build_matcher(
    descriptions_iter: Iterable[tuple], active_concepts: set
) -> KeywordProcessor:
    """Build a case-insensitive FlashText matcher mapping every active English
    description -> conceptId. Short terms (< MIN_TERM_LEN) are skipped to keep noise
    down. Duplicate (concept, term) pairs are deduped so we add each only once."""
    kp = KeywordProcessor(case_sensitive=False)
    added = 0
    skipped_short = 0
    seen: set = set()
    for cid, term, _type_id in descriptions_iter:
        if cid not in active_concepts:
            continue
        clean = strip_semantic_tag(term)
        if len(clean) < MIN_TERM_LEN:
            skipped_short += 1
            continue
        key = (cid, clean.lower())
        if key in seen:
            continue
        seen.add(key)
        # clean_name = conceptId so extract_keywords() returns the concept directly.
        kp.add_keyword(clean, cid)
        added += 1
    print(
        f"Matcher built: {added:,} (term, concept) entries added; "
        f"{skipped_short:,} short terms skipped."
    )
    return kp

## 5. Markdown processor

Splits each `.md` file into pages on top-level (`#` or `##`) headings, tokenises every page with a blank spaCy English tokenizer (no model download), and assigns each tagged token a `(tag, entity_id, concept_id)` triple based on which FlashText match (if any) contains it. Only tagged tokens produce CSV rows.

In [ ]:
import spacy

# Blank English tokenizer: rules-only, no model weights, fast and small.
_NLP = spacy.blank("en")

# A page starts at every `# ` or `## ` heading at the start of a line.
_PAGE_BREAK_RE = re.compile(r"^#{1,2}[ \t]", re.MULTILINE)


def split_pages(md_text: str) -> list:
    """Split Markdown into 'pages' on top-level (# or ##) headings."""
    starts = [m.start() for m in _PAGE_BREAK_RE.finditer(md_text)]
    if not starts or starts[0] != 0:
        starts = [0] + starts
    starts.append(len(md_text))
    return [md_text[a:b] for a, b in zip(starts, starts[1:])]


def tokenize_with_spans(text: str):
    """Yield (token_text, start_char, end_char) for every non-whitespace, non-punctuation token."""
    doc = _NLP.make_doc(text)
    for tok in doc:
        if tok.is_space or tok.is_punct:
            continue
        yield tok.text, tok.idx, tok.idx + len(tok)


def tag_page(
    text: str, kp: KeywordProcessor, concept_to_tag: dict, start_entity_id: int
):
    """Tag a single page. Returns (rows, next_entity_id) where rows is a list of
    (token_start, word, tag, entity_id, concept_id) in document order."""
    tokens = list(tokenize_with_spans(text))
    # extract_keywords(span_info=True) -> [(clean_name=conceptId, start, end), ...]
    matches = kp.extract_keywords(text, span_info=True)
    matches.sort(key=lambda m: m[1])

    rows = []
    entity_id = start_entity_id
    for clean_name, m_start, m_end in matches:
        concept_id = clean_name
        tag = concept_to_tag.get(concept_id, DEFAULT_TAG)
        entity_id += 1
        for word, t_start, t_end in tokens:
            if t_start >= m_start and t_end <= m_end:
                rows.append((t_start, word, tag, entity_id, concept_id))
    rows.sort(key=lambda r: (r[0], r[3]))
    return rows, entity_id


def process_md_file(md_text: str, kp: KeywordProcessor, concept_to_tag: dict) -> list:
    """Process one Markdown document. Returns a list of row dicts ready for pandas."""
    pages = split_pages(md_text)
    out = []
    number = 0
    entity_id = 0
    for page_idx, page_text in enumerate(pages, start=1):
        page_rows, entity_id = tag_page(page_text, kp, concept_to_tag, entity_id)
        for _t_start, word, tag, eid, cid in page_rows:
            number += 1
            out.append(
                {
                    "number": number,
                    "word": word,
                    "tag": tag,
                    "page": page_idx,
                    "entity_id": eid,
                    "concept_id": cid,
                }
            )
    return out

## 6. Sanity check (optional but recommended)

Runs the whole pipeline against a tiny in-memory mock SNOMED and a hardcoded paragraph to make sure the matcher + classifier + page splitter all behave correctly **before** you spend time uploading a multi-hundred-MB SNOMED release.

In [ ]:
def _sanity_check():
    """Verify the full pipeline against a hand-built mini-SNOMED.

    Hierarchy used (NOT real SNOMED IDs in every case; the test only proves that
    the priority resolution and matcher behave correctly):

        404684003 Clinical finding
        |-- 64572001 Disorder         <- 'disease'
        |     `-- 22298006 Myocardial infarction
        `-- 39579001 Anaphylaxis      <- 'clinical finding'
        71388002 Procedure            <- 'procedure'
        |   `-- 80146002 Appendectomy
        105590001 Substance           <- 'drug'
        |   `-- 387458008 Aspirin
        80891009 Heart structure      <- falls back to 'clinical term'
    """
    parent_to_children = defaultdict(
        set,
        {
            "404684003": {"39579001", "64572001"},
            "64572001":  {"22298006"},
            "71388002":  {"80146002"},
            "105590001": {"387458008"},
        },
    )
    concept_to_tag_local = build_concept_to_tag(parent_to_children)

    mock_descriptions = [
        ("22298006",  "Myocardial infarction (disorder)", TYPE_FSN),
        ("22298006",  "MI - Myocardial infarction",       TYPE_SYNONYM),
        ("387458008", "Aspirin (substance)",              TYPE_FSN),
        ("387458008", "Acetylsalicylic acid",             TYPE_SYNONYM),
        ("80146002",  "Appendectomy (procedure)",         TYPE_FSN),
        ("39579001",  "Anaphylaxis (finding)",            TYPE_FSN),
        ("80891009",  "Heart structure (body structure)", TYPE_FSN),
    ]
    active = {c for c, *_ in mock_descriptions}
    kp_local = build_matcher(iter(mock_descriptions), active)

    md = (
        "# Case 1\n"
        "Patient with acute myocardial infarction treated with aspirin.\n"
        "\n"
        "## Case 2\n"
        "Anaphylaxis after appendectomy; heart structure intact.\n"
    )
    rows = process_md_file(md, kp_local, concept_to_tag_local)

    print(f"\nSanity-check produced {len(rows)} tagged-token rows:")
    for r in rows:
        print(
            f"  #{r['number']:>2}  page={r['page']}  "
            f"word={r['word']!r:25s} tag={r['tag']:15s} "
            f"entity={r['entity_id']} concept={r['concept_id']}"
        )

    def _has(tag, word):
        return any(r["tag"] == tag and r["word"].lower() == word for r in rows)

    assert _has("disease", "myocardial"),        "MI not classified as disease"
    assert _has("disease", "infarction"),        "MI second token not classified as disease"
    assert _has("drug", "aspirin"),              "Aspirin not classified as drug"
    assert _has("procedure", "appendectomy"),    "Appendectomy not classified as procedure"
    assert _has("clinical finding", "anaphylaxis"), "Anaphylaxis not classified as clinical finding"
    assert any(r["tag"] == "clinical term" and r["word"].lower() in ("heart", "structure")
               for r in rows), "Heart structure fallback (clinical term) failed"

    pages = {r["page"] for r in rows}
    assert pages == {1, 2}, f"Expected pages 1 and 2, got {pages}"

    entities_by_concept = {}
    for r in rows:
        entities_by_concept.setdefault(r["concept_id"], set()).add(r["entity_id"])
    assert len(entities_by_concept["22298006"]) == 1, "MI should be one entity (two words sharing entity_id)"

    print("\nAll sanity checks passed.")


_sanity_check()

## 7. Upload your inputs

Run the next two cells and use the file pickers to upload:

1. A SNOMED CT RF2 release ZIP (e.g. `SnomedCT_InternationalRF2_PRODUCTION_20240101T120000Z.zip`).
2. One or more `.md` files (hold *Ctrl* / *Cmd* to multi-select).

If you are **not** running on Colab, place the files manually instead:

* the SNOMED ZIP anywhere directly under `/content/` (e.g. `/content/snomed.zip`),
* and `.md` files in `/content/markdown_inputs/`.

In [ ]:
# Step 7a: SNOMED CT RF2 release ZIP.
snomed_zip_path = None
try:
    from google.colab import files as _colab_files
    print("Select your SNOMED CT RF2 release ZIP (e.g. SnomedCT_InternationalRF2_PRODUCTION_*.zip):")
    _uploaded = _colab_files.upload()
    if not _uploaded:
        raise RuntimeError("No file uploaded.")
    _name = next(iter(_uploaded))
    snomed_zip_path = Path("/content") / _name
    if not snomed_zip_path.exists():
        # Some Colab runtimes drop files in the cwd instead of /content/.
        snomed_zip_path = Path.cwd() / _name
    print(f"\nReceived: {snomed_zip_path}  ({snomed_zip_path.stat().st_size / 1e6:,.1f} MB)")
except ImportError:
    _zips = sorted(Path("/content").glob("*.zip"))
    if not _zips:
        raise FileNotFoundError(
            "Not running on Colab and no .zip found in /content/. "
            "Place a SNOMED RF2 ZIP there first."
        )
    snomed_zip_path = _zips[0]
    print(f"Using local ZIP: {snomed_zip_path}")

In [ ]:
# Step 7b: one or more .md files.
md_paths: list = []
try:
    from google.colab import files as _colab_files
    print("Select one or more .md files (hold Ctrl/Cmd to multi-select):")
    _uploaded = _colab_files.upload()
    for _name in _uploaded:
        # Try the documented Colab landing dir first, then cwd as a fallback.
        for _candidate in (Path("/content") / _name, Path.cwd() / _name):
            if _candidate.exists():
                _src = _candidate
                break
        else:
            print(f"  (could not find uploaded file on disk: {_name})")
            continue
        if _src.suffix.lower() != ".md":
            print(f"  Skipping non-markdown file: {_name}")
            continue
        _dst = MD_DIR / _name
        _dst.write_bytes(_src.read_bytes())
        md_paths.append(_dst)
    print(f"\n{len(md_paths)} markdown file(s) staged in {MD_DIR}.")
except ImportError:
    md_paths = sorted(MD_DIR.glob("*.md"))
    if not md_paths:
        raise FileNotFoundError(
            f"Not running on Colab and no .md files found in {MD_DIR}. "
            "Drop your files there first."
        )
    print(f"Using {len(md_paths)} local .md files in {MD_DIR}.")

assert md_paths, "No .md files were uploaded."

## 8. Run the pipeline and download the CSVs

Unzips the SNOMED release, builds the hierarchy and matcher, then tags every uploaded `.md` file and writes one `<stem>.tagged.csv` per input. All CSVs are bundled into `tagged_outputs.zip`, which is downloaded automatically on Colab.

For a full SNOMED CT International release this typically takes **5&ndash;10 minutes** end to end on Colab's free tier (most of it spent in the unzip + concept/description scan).

In [ ]:
import pandas as pd
from tqdm.auto import tqdm

# 1. Unzip SNOMED.
print(f"Unzipping {snomed_zip_path.name} -> {SNOMED_DIR} ...")
_t0 = time.time()
with zipfile.ZipFile(snomed_zip_path) as _zf:
    _zf.extractall(SNOMED_DIR)
print(f"  done in {time.time() - _t0:.1f}s")

# 2. Locate and parse the three Snapshot files.
rf2 = find_rf2_files(SNOMED_DIR)
active_concepts = load_active_concept_ids(rf2["concepts"])
parent_to_children = load_isa_edges(rf2["relationships"], active_concepts)
concept_to_tag = build_concept_to_tag(parent_to_children)

# 3. Build the matcher from the active English descriptions (streaming).
print("Building matcher from active English descriptions ...")
_t0 = time.time()
kp = build_matcher(
    iter_active_descriptions(rf2["descriptions"], active_concepts),
    active_concepts,
)
print(f"  done in {time.time() - _t0:.1f}s")

# 4. Tag every uploaded .md file.
CSV_COLUMNS = ["number", "word", "tag", "page", "entity_id", "concept_id"]
written = []
for md_path in tqdm(md_paths, desc="Tagging files"):
    md_text = md_path.read_text(encoding="utf-8", errors="replace")
    rows = process_md_file(md_text, kp, concept_to_tag)
    csv_path = OUT_DIR / f"{md_path.stem}.tagged.csv"
    if rows:
        pd.DataFrame(rows, columns=CSV_COLUMNS).to_csv(csv_path, index=False, encoding="utf-8")
    else:
        # Always write the file (even if empty) so users can see it was processed.
        pd.DataFrame(columns=CSV_COLUMNS).to_csv(csv_path, index=False, encoding="utf-8")
    print(f"  {md_path.name}: {len(rows):,} tagged-token rows -> {csv_path.name}")
    written.append(csv_path)

# 5. Bundle into a single ZIP.
zip_out = OUT_DIR / "tagged_outputs.zip"
with zipfile.ZipFile(zip_out, "w", compression=zipfile.ZIP_DEFLATED) as _zf:
    for p in written:
        _zf.write(p, arcname=p.name)
print(f"\nWrote {len(written)} CSV(s) and bundled them into {zip_out} ({zip_out.stat().st_size / 1e3:,.1f} KB)")

# 6. Trigger browser download on Colab.
try:
    from google.colab import files as _colab_files
    _colab_files.download(str(zip_out))
except ImportError:
    print(f"(Not on Colab; copy the bundle from {zip_out})")

## 9. Preview the first tagged CSV

Shows the first few rows and the tag distribution of the first CSV produced. Helps you sanity-check the run without leaving the notebook.

In [ ]:
if written:
    preview_df = pd.read_csv(written[0])
    print(f"Preview of {written[0].name} ({len(preview_df):,} rows):")
    try:
        from IPython.display import display
        display(preview_df.head(40))
    except Exception:
        print(preview_df.head(40).to_string(index=False))
    print("\nTag distribution:")
    print(preview_df["tag"].value_counts().to_string())